# Example: Exact-Kpoint Approach (InAs)

This notebook demonstrates the `exact_kpoint` approach for computing Auger recombination coefficients using InAs VASP data from the `test-files/` directory.

The `exact_kpoint` approach determines the 4th state at the exact momentum-conserving k-point by running NSCF calculations. This provides higher accuracy than the `nearest_kpoint` approach and does **not** require `ISYM = -1` in the SCF INCAR.

The workflow has an additional stage: generating the NSCF k-point list, running NSCF VASP jobs, and then creating pairs from the NSCF results.

---

In [1]:
import sys, os

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

from auger import AugerCalculator, utilities
import numpy as np

## Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────
VASP_FOLDER = "../test-files/InAs/exact-kpoint-scf-10"
RESULTS_DIR = "./results/InAs/exact_kpoint"

# ── Physical parameters ────────────────────────────────────────────────────
TEMPERATURE    = 300                   # K
DOPING         = 0                     # cm⁻³ (intrinsic)
EXCESS_CARRIER = 1e17                  # cm⁻³

# ── Material parameters (InAs) ─────────────────────────────────────────────
DIELECTRIC     = 15.15                 # Static dielectric constant
BAND_GAP       = 0.4                   # Experimental band gap (eV) at room temperature

# ── Energy windows ─────────────────────────────────────────────────────────
CB_WINDOW      = 0.3                   # eV above CBM
VB_WINDOW      = 0.3                   # eV below VBM

---
## Step 0 — Create calculator and assign bands

In [3]:
calc = AugerCalculator(T=TEMPERATURE, nd=DOPING)

# InAs: first CB is band 9, last VB is band 8 (0-based)
calc.assign_firstCB_and_lastVB(firstCB_index=9, lastVB_index=8)


                              AUGER RECOMBINATION CALCULATOR                              
│ Temperature:                   300 K                                                   │
│ Doping Concentration:          0.00e+00 cm⁻³                                           │
│ Doping Type:                   intrinsic                                               │

  First CB index: 9  |  Last VB index: 8


---
## Step 1 — Parse VASP data

In [5]:
calc.parse_BS_data(
    folder_path=VASP_FOLDER,
    write_path=RESULTS_DIR,
    force_gap=BAND_GAP,
)


──────────────────────────────────────────────────────────────────────────────────────────
📂 Parsing VASP data from: ../test-files/InAs/exact-kpoint-scf-10
──────────────────────────────────────────────────────────────────────────────────────────
  Material: InAs  |  Band gap: 0.0 eV
  Applying scissor shift: +0.4000 eV
  Saved to: ./results/InAs/exact_kpoint/
──────────────────────────────────────────────────────────────────────────────────────────



---
## Step 2 — Import parsed data

In [6]:
calc.import_parsed_BS_data(from_folder=RESULTS_DIR)


──────────────────────────────────────────────────────────────────────────────────────────
📥 Importing from: ./results/InAs/exact_kpoint
──────────────────────────────────────────────────────────────────────────────────────────
  Material: InAs
  K-grid:   (10, 10, 10)  (47 irreducible k-points)
  Bands:    16  |  Gap: 0.4000 eV
──────────────────────────────────────────────────────────────────────────────────────────



---
## Step 3 — Carrier concentrations

In [7]:
fn, fp = calc.calculate_carrier_concentrations(delta_n=EXCESS_CARRIER)

print(f"n  = {calc.n:.4e} cm⁻³")
print(f"p  = {calc.p:.4e} cm⁻³")
print(f"ni = {calc.ni:.4e} cm⁻³")


──────────────────────────────────────────────────────────────────────────────────────────
⚙  Calculating carrier concentrations …
──────────────────────────────────────────────────────────────────────────────────────────

  NON-EQUILIBRIUM CARRIER CONCENTRATIONS
    n = 1.2279e+17 cm⁻³   |   p = 1.2279e+17 cm⁻³
    Efn = 0.2537 eV    |   Efp = 0.1665 eV
    ni  = 2.2790e+16 cm⁻³
  ⏱  00h 00m 00s
──────────────────────────────────────────────────────────────────────────────────────────

n  = 1.2279e+17 cm⁻³
p  = 1.2279e+17 cm⁻³
ni = 2.2790e+16 cm⁻³


---
## Step 4 — Energy-window selection (optional)

In [8]:
CB_auto, VB_auto = calc.calculate_energy_cutoffs(charge_threshold=0.99)

print(f"Auto CB_window = {CB_auto:.4f} eV")
print(f"Auto VB_window = {VB_auto:.4f} eV")

  Energy windows (99% of carriers):
    CB: 0.4000 → 0.4088 eV  (width 0.0088 eV)
    VB: -0.1696 → 0.0000 eV  (width 0.1696 eV)
Auto CB_window = 0.0088 eV
Auto VB_window = 0.1696 eV


---
## Step 5 — Pair generation (EEH) — Exact-kpoint approach

### Stage 1: Generate the NSCF k-point list

This identifies which off-grid k-points need NSCF calculations.

In [9]:
calc.create_exact_kpoint_list(
    CB_window=CB_WINDOW, # or use CB_auto
    VB_window=VB_WINDOW, # or use VB_auto
    auger_type="eeh",
    poscar_path=VASP_FOLDER + "/POSCAR",
    search_mode="Brute_Force",
    num_kpoints="all",
)


                                    EXACT K-POINT LIST                                    
  Expanded k-points: 47 irr → 1000 full BZ  (mesh 10×10×10)

  Initializing energy states …
    Type: EEH  |  Approach: exact_kpoint  |  Search: Brute_Force
    E1(e): 1  |  E3(e): 1  |  E4(h): 31  |  combos: 31
  Generating exact k-point list … (search: Brute_Force)
  Generated 31 k-points
  Saved 31 k-points → exact_kpoints_eeh_47.csv



### Stage 2: Create NSCF input directories

This creates VASP input files for the NSCF calculations at the exact k-points.

In [11]:
# Find the exact_kpoints CSV that was generated in Stage 1
import glob
exact_csv = glob.glob(RESULTS_DIR + "/exact_kpoints_eeh_*.csv")
print(f"Found exact k-points CSV: {exact_csv}")

utilities.create_nscf_inputs(
    scf_folder=VASP_FOLDER,
    nscf_folder=RESULTS_DIR + "/NSCF_eeh_inputs",
    exact_kpoints_table=exact_csv,
    auger_type="eeh",
    num_kpoints_per_file="all",
)

Found exact k-points CSV: ['./results/InAs/exact_kpoint\\exact_kpoints_eeh_47.csv']

────────────────────────────────────────────────────────────────────────────────
Creating NSCF input files:
  Total unique k-points:  42
    Unique target k-pts:  21
    Unique SCF k-pts:     21
  Number of folders:      1
────────────────────────────────────────────────────────────────────────────────

  ⚠  Could not copy 'POTCAR' to folder 1
  ✓ Created folder 1/1: ./results/InAs/exact_kpoint/NSCF_eeh_inputs/NSCF_eeh_1/
    K-points in this folder: 42

✓ NSCF input creation complete!
────────────────────────────────────────────────────────────────────────────────



**At this point, you would submit the NSCF VASP jobs on your HPC cluster.**

For this example, the NSCF results are assumed to be available. In a real workflow, you would:
1. Navigate to the generated NSCF directories.
2. Submit VASP jobs for each.
3. Return here once all jobs have completed.
4. Re-do only step 0 and step 2 then jump to the next stage. 

### Stage 3: Create pairs using NSCF results

In [13]:
# Point to the folder(s) containing completed NSCF VASP outputs
number_of_nscf_files = 1
NSCF_FOLDERS = [f"{RESULTS_DIR}/NSCF_eeh_inputs/NSCF_eeh_{i+1}" for i in range(number_of_nscf_files)]

gen_eeh = calc.create_auger_pairs(
    CB_window=CB_WINDOW,
    VB_window=VB_WINDOW,
    auger_type="eeh",
    approach="exact_kpoint",
    search_mode="Brute_Force",
    nscf_folders=NSCF_FOLDERS,
    num_top_pairs="all",
    exact_kpoints_csv=exact_csv,
)


                                  AUGER PAIR GENERATION                                   
  Type: EEH  |  Approach: exact_kpoint  |  Search: Brute_Force
  CB window: 0.3000 eV  |  VB window: 0.3000 eV
──────────────────────────────────────────────────────────────────────────────────────────
  Reading exact k-point CSV: ['./results/InAs/exact_kpoint\\exact_kpoints_eeh_47.csv']
    Built 31 pairs from exact k-point CSV

  Pair creation complete: 31 pairs
  Saved 31 pairs → auger_eeh_pairs_47_*.csv

  ✓ 31 pairs  |  ⏱  00h 00m 00s



---
## Step 6 — Matrix elements (EEH)

For the `exact_kpoint` approach, you need to provide WAVECAR files from **both** the SCF and NSCF calculations.

In [14]:
# Collect all WAVECAR files (SCF + NSCF)
wavecar_files = [f"{RESULTS_DIR}/NSCF_eeh_inputs/NSCF_eeh_{i+1}/WAVECAR" for i in range(number_of_nscf_files)]  # Start with the first NSCF WAVECAR

print(f"WAVECAR files: {wavecar_files}")

me_eeh = calc.calculate_matrix_elements(
    auger_type="eeh",
    wavecar_files=wavecar_files,
    dielectric_constant=DIELECTRIC,
    num_matrix_elements="all",
)

WAVECAR files: ['./results/InAs/exact_kpoint/NSCF_eeh_inputs/NSCF_eeh_1/WAVECAR']
  Inverse Debye length: 0.010652 Å⁻¹

  Computing 31 matrix elements on 15 cores …
    [    31/31] 100.0%  avg |M|²=1.3452e+00 eV²
  Saved → ./results/InAs/exact_kpoint\eeh_matrix_elements_47.jsonl
  ⏱  00h 01m 54s


---
## Step 7 — Auger coefficients (EEH)

In [15]:
results_eeh = calc.calculate_auger_rates(
    auger_type="eeh",
    delta_function=("Gaussian", "Lorentzian", "Rectangular"),
    FWHM=(0.001, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2),
)


                              AUGER COEFFICIENT CALCULATION                               
  Type: EEH  |  T = 300 K
──────────────────────────────────────────────────────────────────────────────────────────

  ✓ Saved Auger_coefficients_eeh_47.csv
    Gaussian (FWHM=0.05):  C = 5.0281e-98 cm⁶/s
    Lorentzian (FWHM=0.05):  C = 6.5591e-32 cm⁶/s
    Rectangular (FWHM=0.05):  C = 0.0000e+00 cm⁶/s
  ⏱  00h 00m 00s



In [16]:
import pandas as pd

df_eeh = pd.DataFrame(results_eeh)
print("EEH Auger Coefficients (C_n) for InAs [exact_kpoint]")
print("=" * 65)
for delta in df_eeh["Delta function"].unique():
    print(f"\n  {delta}:")
    sub = df_eeh[df_eeh["Delta function"] == delta]
    for _, row in sub.iterrows():
        print(f"    FWHM = {row['FWHM']:.3f} eV  ->  C_n = {row['Auger coefficient']:.4e} cm^6/s")

EEH Auger Coefficients (C_n) for InAs [exact_kpoint]

  Gaussian:
    FWHM = 0.001 eV  ->  C_n = 0.0000e+00 cm^6/s
    FWHM = 0.005 eV  ->  C_n = 0.0000e+00 cm^6/s
    FWHM = 0.010 eV  ->  C_n = 0.0000e+00 cm^6/s
    FWHM = 0.030 eV  ->  C_n = 7.9508e-220 cm^6/s
    FWHM = 0.050 eV  ->  C_n = 5.0281e-98 cm^6/s
    FWHM = 0.070 eV  ->  C_n = 1.4922e-64 cm^6/s
    FWHM = 0.100 eV  ->  C_n = 7.5647e-47 cm^6/s
    FWHM = 0.150 eV  ->  C_n = 1.7224e-37 cm^6/s
    FWHM = 0.200 eV  ->  C_n = 2.8083e-34 cm^6/s

  Lorentzian:
    FWHM = 0.001 eV  ->  C_n = 1.3175e-33 cm^6/s
    FWHM = 0.005 eV  ->  C_n = 6.5874e-33 cm^6/s
    FWHM = 0.010 eV  ->  C_n = 1.3173e-32 cm^6/s
    FWHM = 0.030 eV  ->  C_n = 3.9464e-32 cm^6/s
    FWHM = 0.050 eV  ->  C_n = 6.5591e-32 cm^6/s
    FWHM = 0.070 eV  ->  C_n = 9.1447e-32 cm^6/s
    FWHM = 0.100 eV  ->  C_n = 1.2950e-31 cm^6/s
    FWHM = 0.150 eV  ->  C_n = 1.9018e-31 cm^6/s
    FWHM = 0.200 eV  ->  C_n = 2.4634e-31 cm^6/s

  Rectangular:
    FWHM = 0.001 eV 

---
## Repeat for EHH ($C_p$)

The same multi-stage exact_kpoint workflow applies for the EHH type. Change `auger_type` to `"ehh"` and repeat Steps 5–7.

In [ ]:
# ── Stage 1: Generate exact k-points for EHH ──────────────────────
# calc.create_exact_kpoint_list(
#     CB_window=CB_WINDOW, VB_window=VB_WINDOW,
#     auger_type="ehh",
#     poscar_path=VASP_FOLDER + "/POSCAR",
#     search_mode="Brute_Force",
#     num_kpoints="all",
# )
#
# ── Stage 2: Create NSCF inputs for EHH ────────────────────────────
# exact_csv_ehh = glob.glob(RESULTS_DIR + "/exact_kpoints_ehh_*.csv")
# utilities.create_nscf_inputs(
#     scf_folder=VASP_FOLDER,
#     nscf_folder=RESULTS_DIR + "/NSCF_ehh_inputs",
#     exact_kpoints_table=exact_csv_ehh,
#     auger_type="ehh",
# )
#
# ── (Run NSCF VASP jobs for EHH k-points) ────────────────────────
#
# ── Stage 3: Create EHH pairs ───────────────────────────────────────
# NSCF_FOLDERS_EHH = [RESULTS_DIR + "/NSCF_ehh_inputs"]
# gen_ehh = calc.create_auger_pairs(
#     CB_window=CB_WINDOW, VB_window=VB_WINDOW,
#     auger_type="ehh", approach="exact_kpoint",
#     search_mode="Brute_Force",
#     nscf_folders=NSCF_FOLDERS_EHH,
#     exact_kpoints_csv=exact_csv_ehh,
# )
#
# ── Step 6: Matrix elements for EHH ────────────────────────────────
# me_ehh = calc.calculate_matrix_elements(
#     auger_type="ehh",
#     wavecar_files=wavecar_files,  # reuse SCF + NSCF WAVECARs
#     dielectric_constant=DIELECTRIC,
# )
#
# ── Step 7: Auger coefficients for EHH ─────────────────────────────
# results_ehh = calc.calculate_auger_rates(
#     auger_type="ehh",
#     delta_function=("Gaussian", "Lorentzian", "Rectangular"),
#     FWHM=(0.001, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2),
# )